In [90]:
import pandas as pd
import numpy as np
import networkx as nx
import time
from algorithms import spnsa, _single_source_shortest_path_basic, calculate_reliance_from_sources, draw_graph
import random

In [91]:
import importlib
import algorithms              # make sure the module itself is imported
importlib.reload(algorithms)   # reloads algorithms.py from disk

from algorithms import spnsa, _single_source_shortest_path_basic, calculate_reliance_from_sources, draw_graph

In [92]:


def create_criminal_graph(file_path:str):
    #G = nx.MultiDiGraph()
    G = nx.DiGraph()

    with open(file_path) as f:
        lines = f.readlines()
        for line in lines[1:]:
            p = [x.strip() for x in line.split(',')]
            if p[10] == '1':
                v = p[2]
                w = p[4]
                G.add_edge(v, w)

    f.close()
    return G

def create_graph(file_path:str):

    #G = nx.MultiDiGraph()
    G = nx.DiGraph()

    with open(file_path) as f:
        lines = f.readlines()
        for line in lines[1:]:
            p = [x.strip() for x in line.split(',')]
            v = p[2]
            w = p[4]
            G.add_edge(v, w)

    f.close()
    return G

def get_suspicious_nodes(file_path:str):
    suspicious_nodes = set()

    with open(file_path) as f:
        lines = f.readlines()
        for line in lines[1:]:
            p = [x.strip() for x in line.split(',')]
            if p[10] == '1':
                suspicious_nodes.add(p[2])
                suspicious_nodes.add(p[4])

    f.close()
    return suspicious_nodes

def get_suspicious_edges(file_path:str):
    suspicious_edges = set()

    with open(file_path) as f:
        lines = f.readlines()
        for line in lines[1:]:
            p = [x.strip() for x in line.split(',')]
            if p[10] == '1':
                suspicious_edges.add((p[2], p[4]))

    f.close()
    return suspicious_edges

def find_largest_component(
    G: nx.Graph,
    component_type: str = "weak",  # used only if G is directed
) -> nx.Graph:
    """
    Return the largest component as an induced subgraph (copied).

    - If G is undirected: uses connected_components.
    - If G is directed: uses weakly_connected_components by default,
      or strongly_connected_components if component_type="strong".
    """
    if G.number_of_nodes() == 0:
        # preserve graph type
        return G.copy()

    if G.is_directed():
        if component_type not in {"weak", "strong"}:
            raise ValueError("component_type must be 'weak' or 'strong' for directed graphs.")
        comps = (
            nx.weakly_connected_components(G)
            if component_type == "weak"
            else nx.strongly_connected_components(G)
        )
    else:
        comps = nx.connected_components(G)

    largest_cc = max(comps, key=len)
    return G.subgraph(largest_cc).copy()

def get_connected_components(G):
    if nx.number_connected_components(G) == 1:
        return [G]
        
    connected_components_generatorset = nx.connected_components(G)
    connected_components = []
    for component in connected_components_generatorset:
        component_graph = G.subgraph(component).copy()
        connected_components.append(component_graph)
    return connected_components

def get_eccentricity_distribution_list(component_list):
    ''' component_list : List[component] '''
    eccentricity_distribution_list = []
    for component in component_list:
        eccentricity_distribution = nx.eccentricity(component)
        eccentricity_distribution_list.append(eccentricity_distribution)
    return eccentricity_distribution_list

In [93]:
G_all = create_graph('../../datasets/IBM AML/HI-Small_Trans.csv')
print(G_all)

DiGraph with 515080 nodes and 1015736 edges


In [94]:
G_criminal = create_criminal_graph('../../datasets/IBM AML/HI-Small_Trans.csv')
print(G_criminal)

DiGraph with 6357 nodes and 4936 edges


In [95]:
# largest component H
largest_component = find_largest_component(G_all)
print(largest_component)

DiGraph with 372091 nodes and 874017 edges


In [96]:
criminal_nodes = get_suspicious_nodes('../../datasets/IBM AML/HI-Small_Trans.csv')
criminal_nodes_list = list(criminal_nodes)
all_nodes_list = list(largest_component.nodes())
criminal_edges = get_suspicious_edges('../../datasets/IBM AML/HI-Small_Trans.csv')

In [97]:
print(len(criminal_nodes))
print(len(criminal_edges))

6357
4936


In [98]:
print(len(criminal_nodes & largest_component.nodes()))

6342


In [56]:
upto = 10
out_degree = dict(largest_component.out_degree())
out_degree_largest = sorted(out_degree.items(), key=lambda x: x[1], reverse=True)[:upto]
out_degree_largest = dict(out_degree_largest)
out_degree_largest

{'100428660': 14230,
 '1004286A8': 8846,
 '100428978': 1776,
 '1004286F0': 1575,
 '1004289C0': 1432,
 '100428780': 1421,
 '100428810': 1394,
 '1004287C8': 1182,
 '100428A51': 1147,
 '100428738': 1136}

In [57]:
upto = 10
in_degree = dict(largest_component.in_degree())
in_degree_largest = sorted(in_degree.items(), key=lambda x: x[1], reverse=True)[:upto]
in_degree_largest = dict(in_degree_largest)
in_degree_largest

{'100428660': 545,
 '1004286A8': 328,
 '8051A3FA0': 93,
 '808FAF7C0': 90,
 '80B847A90': 85,
 '80F523260': 83,
 '801AF5660': 83,
 '80640FE10': 83,
 '80640B300': 83,
 '8146A9331': 82}

In [58]:
upto = 10

degree_diff = {
    node: abs(in_degree.get(node, 0) - out_degree.get(node, 0))
    for node in largest_component.nodes()
}

degree_diff_largest = sorted(degree_diff.items(), key=lambda x: x[1], reverse=True)[:upto]
degree_diff_largest = dict(degree_diff_largest)
degree_diff_largest

{'100428660': 13685,
 '1004286A8': 8518,
 '100428978': 1701,
 '1004286F0': 1520,
 '1004289C0': 1364,
 '100428780': 1360,
 '100428810': 1336,
 '100428A51': 1133,
 '1004287C8': 1127,
 '100428738': 1085}

In [59]:
len(degree_diff_largest.keys() & in_degree_largest.keys())

2

# Ex0: Uniformly random feed experiments

In [60]:
# feed size 200; radius 1
number_of_suspicious_nodes = []
number_of_nodes = []
number_of_edges = []
for i in range(100):
    feed = random.sample(all_nodes_list, 200)
    SPN, paths = spnsa(largest_component, feed, radius=1)
    num_suspicious_nodes = len(SPN.nodes() & criminal_nodes)
    number_of_suspicious_nodes.append(num_suspicious_nodes)
    number_of_nodes.append(SPN.number_of_nodes())
    number_of_edges.append(SPN.number_of_edges())
    
avg_num_nodes = sum(number_of_nodes) / len(number_of_nodes)
avg_num_edges = sum(number_of_edges) / len(number_of_edges)
avg_num_suspicious_nodes = sum(number_of_suspicious_nodes) / len(number_of_suspicious_nodes)

print(f'DiGraph with average {avg_num_nodes} and {avg_num_edges} edges')
print(f'Number of suspicious nodes in SPN: {avg_num_suspicious_nodes}')
print(f'Criminal ration in SPN: {avg_num_suspicious_nodes / avg_num_nodes}')

DiGraph with average 368.0 and 168.45 edges
Number of suspicious nodes in SPN: 8.83
Criminal ration in SPN: 0.023994565217391305


In [61]:
# feed size 200; radius 2
number_of_suspicious_nodes = []
number_of_nodes = []
number_of_edges = []

for _ in range(10):
    feed = random.sample(all_nodes_list, 200)
    SPN, paths = spnsa(largest_component, feed, radius=2)
    num_suspicious_nodes = len(SPN.nodes() & criminal_nodes)
    number_of_suspicious_nodes.append(num_suspicious_nodes)
    number_of_nodes.append(SPN.number_of_nodes())
    number_of_edges.append(SPN.number_of_edges())
    
avg_num_nodes = sum(number_of_nodes) / len(number_of_nodes)
avg_num_edges = sum(number_of_edges) / len(number_of_edges)
avg_num_suspicious_nodes = sum(number_of_suspicious_nodes) / len(number_of_suspicious_nodes)

print(f'DiGraph with average {avg_num_nodes} and {avg_num_edges} edges')
print(f'Number of suspicious nodes in SPN: {avg_num_suspicious_nodes}')
print(f'Criminal ration in SPN: {avg_num_suspicious_nodes / avg_num_nodes}')

DiGraph with average 448.3 and 250.9 edges
Number of suspicious nodes in SPN: 13.7
Criminal ration in SPN: 0.03055989292884229


In [62]:
# feed size 500; radius 1
number_of_suspicious_nodes = []
number_of_nodes = []
number_of_edges = []

for _ in range(100):
    feed = random.sample(all_nodes_list, 500)
    SPN, paths = spnsa(largest_component, feed, radius=1)
    num_suspicious_nodes = len(SPN.nodes() & criminal_nodes)
    number_of_suspicious_nodes.append(num_suspicious_nodes)
    number_of_nodes.append(SPN.number_of_nodes())
    number_of_edges.append(SPN.number_of_edges())
    
avg_num_nodes = sum(number_of_nodes) / len(number_of_nodes)
avg_num_edges = sum(number_of_edges) / len(number_of_edges)
avg_num_suspicious_nodes = sum(number_of_suspicious_nodes) / len(number_of_suspicious_nodes)

print(f'DiGraph with average {avg_num_nodes} and {avg_num_edges} edges')
print(f'Number of suspicious nodes in SPN: {avg_num_suspicious_nodes}')
print(f'Criminal ration in SPN: {avg_num_suspicious_nodes / avg_num_nodes}')

DiGraph with average 919.32 and 422.13 edges
Number of suspicious nodes in SPN: 24.41
Criminal ration in SPN: 0.026552234260105292


# Ex1: Largest degree difference feed experiments

In [63]:
# feed size 200; radius 1
feed = dict(sorted(degree_diff.items(), key=lambda x: x[1], reverse=True)[:200])
SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 255 nodes and 134 edges
Number of suspicious nodes in SPN: 35
Criminal ration in SPN: 0.13725490196078433


In [64]:
# feed size 200; radius 2
feed = dict(sorted(degree_diff.items(), key=lambda x: x[1], reverse=True)[:200])
SPN, paths = spnsa(largest_component, feed, radius=2)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 326 nodes and 249 edges
Number of suspicious nodes in SPN: 41
Criminal ration in SPN: 0.12576687116564417


In [65]:
# feed size 500; radius 1
feed = dict(sorted(degree_diff.items(), key=lambda x: x[1], reverse=True)[:500])
SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 588 nodes and 320 edges
Number of suspicious nodes in SPN: 66
Criminal ration in SPN: 0.11224489795918367


# Ex2: Non-zero in-degree & zero out-degree experiments (Collect)

In [66]:
upto = 10

nonzero_in_degree = {
    node: in_degree.get(node, 0)
    for node in largest_component.nodes() if out_degree.get(node, 0) == 0
}

nonzero_in_degree_largest = sorted(nonzero_in_degree.items(), key=lambda x: x[1], reverse=True)[:upto]
nonzero_in_degree_largest = dict(nonzero_in_degree_largest)
nonzero_in_degree_largest

{'8051A3FA0': 93,
 '8050E2560': 82,
 '809FD0970': 81,
 '811566170': 80,
 '803382430': 79,
 '807B77130': 75,
 '80352D880': 74,
 '80A10EBE0': 62,
 '80AB6EB90': 61,
 '800C4C1B0': 59}

In [67]:
# feed size 200; radius 1
feed = dict(sorted(nonzero_in_degree.items(), key=lambda x: x[1], reverse=True)[:200])
SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 200 nodes and 0 edges
Number of suspicious nodes in SPN: 14
Criminal ration in SPN: 0.07


In [68]:
# feed size 200; radius 2
feed = dict(sorted(nonzero_in_degree.items(), key=lambda x: x[1], reverse=True)[:200])
SPN, paths = spnsa(largest_component, feed, radius=2)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 200 nodes and 0 edges
Number of suspicious nodes in SPN: 14
Criminal ration in SPN: 0.07


In [69]:
# feed size 500; radius 1
feed = dict(sorted(nonzero_in_degree.items(), key=lambda x: x[1], reverse=True)[:500])
SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 500 nodes and 0 edges
Number of suspicious nodes in SPN: 32
Criminal ration in SPN: 0.064


# Ex3: Non-zero out-degree & zero in-degree experiments (Distribute)

In [70]:
upto = 10

nonzero_out_degree = {
    node: out_degree.get(node, 0)
    for node in largest_component.nodes() if in_degree.get(node, 0) == 0
}

nonzero_out_degree_largest = sorted(nonzero_out_degree.items(), key=lambda x: x[1], reverse=True)[:upto]
nonzero_out_degree_largest = dict(nonzero_out_degree_largest)
nonzero_out_degree_largest

{'8001CA830': 22,
 '8003EC930': 21,
 '80124BDE0': 21,
 '8005E73B0': 20,
 '8003BF640': 19,
 '80065E800': 19,
 '80767B210': 19,
 '8004AB200': 18,
 '800A87960': 17,
 '801BA1830': 17}

In [71]:
# feed size 200; radius 1
feed = dict(sorted(nonzero_out_degree.items(), key=lambda x: x[1], reverse=True)[:200])
SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 486 nodes and 288 edges
Number of suspicious nodes in SPN: 15
Criminal ration in SPN: 0.030864197530864196


In [72]:
# feed size 200; radius 2
feed = dict(sorted(nonzero_out_degree.items(), key=lambda x: x[1], reverse=True)[:200])
SPN, paths = spnsa(largest_component, feed, radius=2)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 715 nodes and 518 edges
Number of suspicious nodes in SPN: 23
Criminal ration in SPN: 0.032167832167832165


In [73]:
# feed size 500; radius 1
feed = dict(sorted(nonzero_out_degree.items(), key=lambda x: x[1], reverse=True)[:500])
SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 1201 nodes and 738 edges
Number of suspicious nodes in SPN: 33
Criminal ration in SPN: 0.027477102414654453


# Ex4: 50% Collect and 50% Distribute feed experiments

In [74]:
# feed size 200; radius 1
upto = 200
nonzero_in_degree_largest = dict(sorted(nonzero_in_degree.items(), key=lambda x: x[1], reverse=True)[:upto//2])
nonzero_out_degree_largest = dict(sorted(nonzero_out_degree.items(), key=lambda x: x[1], reverse=True)[:upto//2])
feed = nonzero_in_degree_largest | nonzero_out_degree_largest

SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')


DiGraph with 345 nodes and 147 edges
Number of suspicious nodes in SPN: 16
Criminal ration in SPN: 0.0463768115942029


In [75]:
# feed size 200; radius 2
upto = 200
nonzero_in_degree_largest = dict(sorted(nonzero_in_degree.items(), key=lambda x: x[1], reverse=True)[:upto//2])
nonzero_out_degree_largest = dict(sorted(nonzero_out_degree.items(), key=lambda x: x[1], reverse=True)[:upto//2])
feed = nonzero_in_degree_largest | nonzero_out_degree_largest

SPN, paths = spnsa(largest_component, feed, radius=2)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')


DiGraph with 467 nodes and 270 edges
Number of suspicious nodes in SPN: 21
Criminal ration in SPN: 0.044967880085653104


In [76]:
# feed size 500; radius 1
upto = 500
nonzero_in_degree_largest = dict(sorted(nonzero_in_degree.items(), key=lambda x: x[1], reverse=True)[:upto//2])
nonzero_out_degree_largest = dict(sorted(nonzero_out_degree.items(), key=lambda x: x[1], reverse=True)[:upto//2])
feed = nonzero_in_degree_largest | nonzero_out_degree_largest

SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')


DiGraph with 852 nodes and 368 edges
Number of suspicious nodes in SPN: 37
Criminal ration in SPN: 0.04342723004694836


# Ex5: High illicit degrees as feed experiments

In [77]:
upto = 10
illicit_degree = dict(G_criminal.degree())
illicit_degree_largest = sorted(illicit_degree.items(), key=lambda x: x[1], reverse=True)[:upto]
illicit_degree_largest = dict(illicit_degree_largest)
illicit_degree_largest

{'100428660': 236,
 '1004286A8': 156,
 '100428978': 29,
 '8075AC7C0': 28,
 '80154AAF0': 27,
 '802E2AD70': 27,
 '100428810': 26,
 '8106B1750': 26,
 '80794B6F0': 26,
 '803DE4A90': 26}

In [78]:
# feed size 200; radius 1
feed = dict(sorted(illicit_degree.items(), key=lambda x: x[1], reverse=True)[:200])
SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 423 nodes and 394 edges
Number of suspicious nodes in SPN: 307
Criminal ration in SPN: 0.7257683215130024


In [79]:
# feed size 200; radius 2
feed = dict(sorted(illicit_degree.items(), key=lambda x: x[1], reverse=True)[:200])
SPN, paths = spnsa(largest_component, feed, radius=2)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 660 nodes and 778 edges
Number of suspicious nodes in SPN: 448
Criminal ration in SPN: 0.6787878787878788


In [80]:
# feed size 500; radius 1
feed = dict(sorted(illicit_degree.items(), key=lambda x: x[1], reverse=True)[:500])
SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 944 nodes and 960 edges
Number of suspicious nodes in SPN: 678
Criminal ration in SPN: 0.7182203389830508


In [81]:
# random 200 feed; radius 1
import random

feed = dict(random.sample(list(illicit_degree.items()), 200))
SPN, paths = spnsa(largest_component, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 385 nodes and 194 edges
Number of suspicious nodes in SPN: 262
Criminal ration in SPN: 0.6805194805194805
